In [ ]:
from langchain.text_splitter import MarkdownHeaderTextSplitter
from langchain.vectorstores import FAISS
import os
from langchain_community.embeddings import OpenAIEmbeddings

In [ ]:
#Read md file
def read_file(file_path):
    with open(file_path, 'r') as file:
        file_content = file.read()
    return file_content


class Document:
    def __init__(self, metadata, page_content):
        self.metadata = metadata
        self.page_content = page_content

    def __repr__(self):
        return f"Document(metadata={self.metadata}, page_content={repr(self.page_content)})"

# Mapping headers to their respective markdown levels
header_mapping = {
    "Header 1": "#",
    "Header 2": "##",
    "Header 3": "###",
    "Header 4": "####",
    "Header 5": "#####"
}

# Function to extract metadata and prepend to page_content with proper headers
def create_new_document_with_prepend_metadata(doc):
    headers = []
    for header, value in doc.metadata.items():
        if header in header_mapping:
            headers.append(f"{header_mapping[header]} {value} \n")
    metadata_str = ''.join(headers)
    new_page_content = metadata_str + doc.page_content
    return Document(doc.metadata, new_page_content)

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ('####', "Header 4"),
    ('#####', "Header 5")
]

In [ ]:
md_folder_path  ='docs'
md_file_path = []

for file_name in os.listdir(md_folder_path):
    if os.path.isfile(os.path.join(md_folder_path, file_name)):
        md_file_path.append(md_folder_path+'/'+file_name)

In [ ]:
#create splitter base on header
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

chunked_md_list = []
chunked_md_list_w_header_incontent= []

for i in md_file_path:
    file_content = read_file(i)
    #split and exclude TOC chunk
    md_header_splits = markdown_splitter.split_text(file_content)
    md_header_splits_exlude_TOC = md_header_splits[1:]
    # Create new documents with metadata included in context
    md_header_in_context = [create_new_document_with_prepend_metadata(doc) for doc in md_header_splits_exlude_TOC]
    # # Char-level splits
    # chunk_size = 15000
    # chunk_overlap = 5000
    # text_splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=chunk_size, chunk_overlap=chunk_overlap
    # )
    # md_header_in_context = text_splitter.split_documents(md_header_in_context)
    # md_header_splits_exlude_TOC = text_splitter.split_documents(md_header_splits_exlude_TOC)
    #get file name and set key name
    filename = os.path.basename(i)
    key_name = 'File Name'
    # Add the new key-value pair to each metadata dictionary
    for chunk in md_header_splits_exlude_TOC:
        chunk.metadata[key_name] = filename
    for chunk in md_header_in_context:
        chunk.metadata[key_name] = filename
    #append all document chunked content
    chunked_md_list_w_header_incontent.extend(md_header_in_context)
    chunked_md_list.extend(md_header_splits_exlude_TOC)
    

In [ ]:
# Extract texts and metadata from your documents
texts = [doc.page_content for doc in chunked_md_list_w_header_incontent]
metadatas = [doc.metadata for doc in chunked_md_list_w_header_incontent]

# Initialize Embeddings
API_KEY = os.environ.get("API_KEY")
EMBEDDING_MODEL = os.environ.get("EMBED_MODEL_NAME", "BAAI/bge-small-en-v1.5")
EMBEDDING_URL = os.environ.get("BASE_URL")
embeddings = OpenAIEmbeddings(model= EMBEDDING_MODEL, base_url=EMBEDDING_URL, api_key = API_KEY, tiktoken_enabled=False) 

# Create Vector Store using texts and metadata
vector_store = FAISS.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas  # Pass metadata separately
)

# Optional: Save the vector store for future use
vector_store.save_local("vector_store")